# Hierarchical Two-Stage Rainfall Forecasting
**Focus**: AWS/IoT Feature Selection, Two-Stage Prediction (Occurrence -> Amount), and Probability Calibration.

## Purpose
This notebook implements a Two-Stage operational forecasting framework. 
Stage 1 focuses purely on Rain Occurrence (maximizing CSI and Recall). 
Stage 2 focuses purely on Rain Amount (mm) and is ONLY trained/evaluated on samples where rain > 0.


In [ ]:
!pip install --upgrade shap
import os, random, json, logging, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import optuna
import xgboost as xgb
import shap

from sklearn.preprocessing import MinMaxScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, confusion_matrix, log_loss, mean_squared_error, 
    mean_absolute_error, r2_score, mean_absolute_percentage_error, 
    precision_recall_curve, roc_curve, auc
)
from sklearn.calibration import calibration_curve

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    logger.info(f"Seed set to {seed}")

seed_everything(42)


# Data Loading (AWS Focused)

In [ ]:
def get_paths():
    cwd = Path.cwd()
    if '/kaggle' in str(cwd) or '\\kaggle' in str(cwd):
        return Path("/kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/open_meteo_jerukagung/cuaca_jerukagung.csv")
    return Path(r"D:\Github\Projek_Rainfall\Analisis_Meteorologi\open_meteo_jerukagung\cuaca_jerukagung.csv")

def load_data(filepath):
    logger.info(f"Loading data from {filepath}")
    df = pd.read_csv(filepath)
    if 'datetime' in df.columns: df = df.set_index('datetime')
    elif 'date' in df.columns: df = df.set_index('date')
    df.index = pd.to_datetime(df.index, utc=True).tz_convert('Asia/Jakarta').tz_localize(None)
    df.index.name = 'date'
    df = df.sort_index()
    
    # AWS/IoT Deployable sensors only
    essential_cols = [
        'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'rain', 
        'wind_speed_10m', 'wind_gusts_10m', 'wind_direction_10m', 'surface_pressure', 
        'sunshine_duration', 'shortwave_radiation', 'wet_bulb_temperature_2m', 'vapour_pressure_deficit'
    ]
    cols_to_keep = [c for c in essential_cols if c in df.columns]
    df = df[cols_to_keep]
    
    if 'rain' in df.columns: df.loc[df['rain'] < 0, 'rain'] = 0
    return df

data_path = get_paths()
df_raw = load_data(data_path)
display(df_raw.head())


# Feature Engineering
## Purpose
Generate precise historical lags, rolling regimes, and atmospheric trends based exclusively on AWS variables.


In [ ]:
def generate_features(df):
    df = df.copy()
    
    # 0. Vectorize Wind (U and V components)
    if 'wind_speed_10m' in df.columns and 'wind_direction_10m' in df.columns:
        # Convert meteorological wind direction to radians
        wd_rad = df['wind_direction_10m'] * np.pi / 180.0
        # Calculate U (East-West) and V (North-South) components
        df['wind_u'] = -df['wind_speed_10m'] * np.sin(wd_rad)
        df['wind_v'] = -df['wind_speed_10m'] * np.cos(wd_rad)
        # Drop original raw wind speed and direction to avoid 360 degree discontinuity
        df = df.drop(columns=['wind_speed_10m', 'wind_direction_10m'])
        
    # 1. Exact Lags
    if 'rain' in df.columns:
        for lag in [1, 2, 3, 6, 12, 24]: df[f'rain_lag_{lag}'] = df['rain'].shift(lag)
    for lag in [1, 3, 6]:
        if 'relative_humidity_2m' in df.columns: df[f'humidity_lag_{lag}'] = df['relative_humidity_2m'].shift(lag)
        if 'surface_pressure' in df.columns: df[f'pressure_lag_{lag}'] = df['surface_pressure'].shift(lag)
        if 'temperature_2m' in df.columns: df[f'temperature_lag_{lag}'] = df['temperature_2m'].shift(lag)
    for lag in [1, 3]:
        if 'wind_u' in df.columns: df[f'wind_u_lag_{lag}'] = df['wind_u'].shift(lag)
        if 'wind_v' in df.columns: df[f'wind_v_lag_{lag}'] = df['wind_v'].shift(lag)
        if 'wind_gusts_10m' in df.columns: df[f'wind_gust_lag_{lag}'] = df['wind_gusts_10m'].shift(lag)
        
    # 2. Exact Trends
    for t in [1, 3]:
        if 'temperature_2m' in df.columns: df[f'temperature_change_{t}h'] = df['temperature_2m'].diff(t)
        if 'relative_humidity_2m' in df.columns: df[f'humidity_change_{t}h'] = df['relative_humidity_2m'].diff(t)
        if 'surface_pressure' in df.columns: df[f'pressure_change_{t}h'] = df['surface_pressure'].diff(t)
    if 'wind_u' in df.columns: 
        df['wind_u_change_1h'] = df['wind_u'].diff(1)
        df['wind_v_change_1h'] = df['wind_v'].diff(1)
        
    # 3. Rolling Features
    windows = [3, 6, 12, 24]
    roll_cols = [c for c in ['rain', 'relative_humidity_2m', 'surface_pressure', 'temperature_2m'] if c in df.columns]
    for col in roll_cols:
        for w in windows:
            df[f'{col}_mean_{w}h'] = df[col].rolling(w, min_periods=1).mean()
            df[f'{col}_std_{w}h'] = df[col].rolling(w, min_periods=1).std().fillna(0)
            df[f'{col}_min_{w}h'] = df[col].rolling(w, min_periods=1).min()
            df[f'{col}_max_{w}h'] = df[col].rolling(w, min_periods=1).max()
            
    df = df.dropna()
    return df

df_feat = generate_features(df_raw)
logger.info(f"Generated {df_feat.shape[1]} features")


# Data Splitting & Targets

In [ ]:
def prepare_targets(df):
    agg_rules = {c: 'mean' for c in df.columns}
    if 'rain' in df.columns: agg_rules['rain'] = 'sum'
        
    df_3h = df.resample('3h').agg(agg_rules).dropna()
    df_3h['target_amount'] = df_3h['rain'].shift(-1)
    df_3h = df_3h.dropna()
    
    df_3h['target_occurrence'] = (df_3h['target_amount'] > 0).astype(int)
    return df_3h

df_processed = prepare_targets(df_feat)
\ntrain_mask = (df_processed.index.year >= 2005) & (df_processed.index.year <= 2023)
val_mask = (df_processed.index.year == 2024)
test_mask = (df_processed.index.year == 2025)

features = df_processed.drop(columns=['target_amount', 'target_occurrence'])
targets = df_processed[['target_amount', 'target_occurrence']]
feature_names = features.columns.tolist()

X_train, y_train = features[train_mask], targets[train_mask]
X_val, y_val = features[val_mask], targets[val_mask]
X_test, y_test = features[test_mask], targets[test_mask]

# Create subsets for Regression Stage (ONLY rain > 0)
train_rain_mask = y_train['target_occurrence'] == 1
val_rain_mask = y_val['target_occurrence'] == 1
test_rain_mask = y_test['target_occurrence'] == 1

X_train_reg, y_train_reg = X_train[train_rain_mask], y_train[train_rain_mask]
X_val_reg, y_val_reg = X_val[val_rain_mask], y_val[val_rain_mask]
X_test_reg, y_test_reg = X_test[test_rain_mask], y_test[test_rain_mask]

scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_reg_s = scaler.transform(X_train_reg)
X_val_reg_s = scaler.transform(X_val_reg)
X_test_reg_s = scaler.transform(X_test_reg)


# Metrics & Diagnostics

In [ ]:
def met_metrics(y_true, y_pred):
    hits = np.sum((y_pred == 1) & (y_true == 1))
    misses = np.sum((y_pred == 0) & (y_true == 1))
    false_alarms = np.sum((y_pred == 1) & (y_true == 0))
    correct_negatives = np.sum((y_pred == 0) & (y_true == 0))
    
    csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else 0
    pod = hits / (hits + misses) if (hits + misses) > 0 else 0
    far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else 0
    
    total = hits + misses + false_alarms + correct_negatives
    hits_random = ((hits + misses) * (hits + false_alarms)) / total if total > 0 else 0
    ets = (hits - hits_random) / (hits + misses + false_alarms - hits_random) if (hits + misses + false_alarms - hits_random) > 0 else 0
    hss = (2 * (hits * correct_negatives - misses * false_alarms)) / ((hits + misses) * (misses + correct_negatives) + (hits + false_alarms) * (false_alarms + correct_negatives)) if total > 0 else 0
    
    return {'CSI': csi, 'POD': pod, 'FAR': far, 'ETS': ets, 'HSS': hss}

def evaluate_models(y_test_occ, prob_occ_uncal, prob_occ_cal, pred_occ_cal, y_test_reg, pred_reg):
    # Regression
    rmse = np.sqrt(mean_squared_error(y_test_reg, pred_reg))
    mae = mean_absolute_error(y_test_reg, pred_reg)
    r2 = r2_score(y_test_reg, pred_reg)
    nse = 1 - (np.sum((y_test_reg - pred_reg)**2) / (np.sum((y_test_reg - np.mean(y_test_reg))**2) + 1e-5))
    
    r_pearson = np.corrcoef(y_test_reg, pred_reg)[0,1]
    kge = 1 - np.sqrt((r_pearson - 1)**2 + (np.std(pred_reg)/(np.std(y_test_reg)+1e-5) - 1)**2 + (np.mean(pred_reg)/(np.mean(y_test_reg)+1e-5) - 1)**2)
    
    # Classification
    met = met_metrics(y_test_occ, pred_occ_cal)
    
    report = {
        'Regression_RainyDaysOnly': {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'NSE': nse, 'KGE': kge},
        'Classification': {
            'Accuracy': accuracy_score(y_test_occ, pred_occ_cal),
            'Precision': precision_score(y_test_occ, pred_occ_cal, zero_division=0),
            'Recall': recall_score(y_test_occ, pred_occ_cal, zero_division=0),
            'F1': f1_score(y_test_occ, pred_occ_cal, zero_division=0),
            'ROC_AUC': roc_auc_score(y_test_occ, prob_occ_cal),
            'Brier_Uncalibrated': brier_score_loss(y_test_occ, prob_occ_uncal),
            'Brier_Calibrated': brier_score_loss(y_test_occ, prob_occ_cal)
        },
        'Meteorological': met
    }
    return report

def plot_calibration(y_true, prob_uncal, prob_cal):
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    f_op_uncal, m_pv_uncal = calibration_curve(y_true, prob_uncal, n_bins=10)
    f_op_cal, m_pv_cal = calibration_curve(y_true, prob_cal, n_bins=10)
    
    plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
    plt.plot(m_pv_uncal, f_op_uncal, "s-", label="Uncalibrated")
    plt.plot(m_pv_cal, f_op_cal, "s-", label="Calibrated (Isotonic)")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title("Reliability Diagram")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    sns.histplot(prob_uncal, color='red', alpha=0.3, label='Uncal', kde=True)
    sns.histplot(prob_cal, color='blue', alpha=0.3, label='Calibrated', kde=True)
    plt.legend()
    plt.title("Probability Distribution")
    plt.tight_layout()
    plt.show()

def plot_classification(y_true, pred, prob):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    sns.heatmap(confusion_matrix(y_true, pred), annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    
    plt.subplot(1, 3, 2)
    fpr, tpr, _ = roc_curve(y_true, prob)
    plt.plot(fpr, tpr, label=f'AUC={auc(fpr, tpr):.3f}')
    plt.plot([0,1],[0,1],'k--')
    plt.legend()
    plt.title('ROC Curve')
    
    plt.subplot(1, 3, 3)
    pr, rc, _ = precision_recall_curve(y_true, prob)
    plt.plot(rc, pr, label=f'AUC={auc(rc, pr):.3f}')
    plt.legend()
    plt.title('Precision-Recall')
    plt.tight_layout()
    plt.show()

def plot_regression(y_true, pred):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.scatter(y_true, pred, alpha=0.3)
    plt.plot([0, max(y_true)], [0, max(y_true)], 'r--')
    plt.xlabel('Observed')
    plt.ylabel('Predicted')
    plt.title('Pred vs Obs (Rainy Only)')
    
    plt.subplot(1, 3, 2)
    sns.histplot(y_true - pred, kde=True)
    plt.title('Residual Distribution')
    
    plt.subplot(1, 3, 3)
    plt.plot(y_true.values[:50], label='Obs')
    plt.plot(pred[:50], label='Pred')
    plt.legend()
    plt.title('Time Series Snippet')
    plt.tight_layout()
    plt.show()


# Bayesian Optuna

In [ ]:
def obj_occ(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'device': 'cuda',
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0)
    }
    clf = xgb.XGBClassifier(**params, random_state=42)
    clf.fit(X_train_s, y_train['target_occurrence'], verbose=False)
    preds = clf.predict_proba(X_val_s)[:, 1]
    return log_loss(y_val['target_occurrence'], preds, labels=[0,1])

def obj_reg(trial):
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda',
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0)
    }
    reg = xgb.XGBRegressor(**params, random_state=42)
    reg.fit(X_train_reg_s, y_train_reg['target_amount'], verbose=False)
    preds = reg.predict(X_val_reg_s)
    return mean_squared_error(y_val_reg['target_amount'], preds)

logger.info("Running XGBoost Optuna (demo trials)...")
study_occ = optuna.create_study(direction='minimize')
study_occ.optimize(obj_occ, n_trials=2)
p_occ = study_occ.best_params
p_occ.update({'objective': 'binary:logistic', 'tree_method': 'hist', 'device': 'cuda'})

study_reg = optuna.create_study(direction='minimize')
study_reg.optimize(obj_reg, n_trials=2)
p_reg = study_reg.best_params
p_reg.update({'objective': 'reg:squarederror', 'tree_method': 'hist', 'device': 'cuda'})


# Training, Calibration, & Validation

In [ ]:
# 1. Train Occurrence Model
clf = xgb.XGBClassifier(**p_occ, random_state=42)
clf.fit(X_train_s, y_train['target_occurrence'], 
        eval_set=[(X_val_s, y_val['target_occurrence'])], verbose=50)

# Uncalibrated Probs
val_prob_uncal = clf.predict_proba(X_val_s)[:, 1]
test_prob_uncal = clf.predict_proba(X_test_s)[:, 1]

# 2. Isotonic Calibration
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(val_prob_uncal, y_val['target_occurrence'])
test_prob_cal = calibrator.predict(test_prob_uncal)
test_pred_cal = (test_prob_cal > 0.5).astype(int)

# 3. Train Regression Model (Only on rainy days)
reg = xgb.XGBRegressor(**p_reg, random_state=42)
reg.fit(X_train_reg_s, y_train_reg['target_amount'], 
        eval_set=[(X_val_reg_s, y_val_reg['target_amount'])], verbose=50)

test_pred_reg = np.maximum(0, reg.predict(X_test_reg_s))

# 4. Evaluation
report = evaluate_models(y_test['target_occurrence'], test_prob_uncal, test_prob_cal, test_pred_cal, y_test_reg['target_amount'], test_pred_reg)
print(json.dumps(report, indent=4))

plot_calibration(y_test['target_occurrence'], test_prob_uncal, test_prob_cal)
plot_classification(y_test['target_occurrence'], test_pred_cal, test_prob_cal)
plot_regression(y_test_reg['target_amount'], test_pred_reg)


# SHAP Explainability

In [ ]:
explainer = shap.TreeExplainer(clf)
shap_val = explainer(X_test_s) # Returns Explanation object

plt.figure()
shap.plots.summary(shap_val, max_display=20, show=False)
plt.title("SHAP Summary")
plt.show()

plt.figure()
shap.plots.bar(shap_val, max_display=10, show=False)
plt.title("Top 10 Variables (Bar)")
plt.show()

plt.figure()
shap.plots.waterfall(shap_val[0], show=False)
plt.title("SHAP Waterfall (Sample 0)")
plt.show()

# Print Top Variables
vals = np.abs(shap_val.values).mean(0)
feature_importance = pd.DataFrame(list(zip(feature_names, vals)), columns=['Feature', 'Importance']).sort_values(by='Importance', ascending=False)
print("TOP 20 VARIABLES:\n", feature_importance.head(20))


# Documentation Report: Explainability
## Model Understanding
Through SHAP analysis and feature importance weights, we can isolate the operational value of AWS deployment:
- **Top Dominant Variables**: Typically, `rain_lag` and `humidity_change` demonstrate the highest predictive capability. Shortwave radiation provides critical context for diurnal convective cycles.
- **Why Two-Stage?**: Training the regressor exclusively on rainy days fundamentally changes the loss landscape. The model no longer needs to squelch predictions to zero on dry days, leading to far more accurate peak-storm millimeter estimations.
- **Why Calibration?**: Uncalibrated outputs are mathematically bound to the training distribution, not operational probabilities. Isotonic regression forces the binary classifier's confidence (e.g. 80%) to map to real-world frequency.
